# Загрузка данных из CSV, Excel и JSON

## Практическая работа

Вы работаете начинающим аналитиком службы поддержки. Данные поступили из трёх систем:

- журнал обращений — `tickets.csv`;
- справочник клиентов — `clients.xlsx`;
- справочник каналов — `channels.json`.

Нужно загрузить файлы, проверить их структуру, привести ключи к совместимому виду, объединить таблицы и сохранить единый аналитический набор данных.

### Результат работы

В папке `outputs` должен появиться файл `support_tickets_datamart.csv`.

## План занятия

| Этап | Время | Результат |
|---|---:|---|
| Постановка задачи | 7 минут | Понимаем, зачем нужны три файла |
| Подготовка среды | 10 минут | Проверяем Python, pandas и папки |
| Загрузка CSV | 15 минут | Получаем таблицу обращений |
| Загрузка XLSX и JSON | 13 минут | Получаем два справочника |
| Подготовка ключей | 10 минут | Убираем пробелы и согласуем типы |
| Объединение | 15 минут | Собираем единый DataFrame |
| Контроль качества | 10 минут | Проверяем строки и несовпавшие ключи |
| Сохранение | 7 минут | Записываем и повторно читаем результат |
| Итоги | 3 минуты | Проходим чек-лист |

## 1. Почему мы работаем с несколькими файлами

В реальных системах данные часто разделены:

- основная таблица хранит события или операции;
- справочники расшифровывают идентификаторы;
- разные подразделения выгружают данные в разных форматах.

Главная задача аналитика — не просто открыть файлы, а убедиться, что их можно корректно связать.

## 2. Подготовка рабочей среды

Выполните следующую ячейку. Если версии отобразились, основные библиотеки доступны.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
from IPython.display import display

print("Версия Python:", sys.version.split()[0])
print("Версия pandas:", pd.__version__)
print("Текущая рабочая папка:", Path.cwd())

### Создаём единые относительные пути

Относительный путь работает внутри папки проекта и не привязан к компьютеру автора.

In [ ]:
# Notebook расположен в корне учебного проекта.
# Все пути относительные, поэтому проект работает и в Google Colab, и в VS Code.
PROJECT_ROOT = Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

RAW_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TICKETS_PATH = RAW_DIR / "tickets.csv"
CLIENTS_PATH = RAW_DIR / "clients.xlsx"
CHANNELS_PATH = RAW_DIR / "channels.json"
OUTPUT_PATH = OUTPUT_DIR / "support_tickets_datamart.csv"

print("Папка исходных данных:", RAW_DIR)
print("Папка результатов:", OUTPUT_DIR)

### Резервное создание данных

В комплекте уже есть файлы. Ячейка ниже нужна на случай, если в Colab был загружен только notebook.

In [ ]:
# Резервный сценарий для Google Colab.
# Если исходные файлы не загружены вместе с notebook, эта ячейка создаст их автоматически.

tickets_rows = [
    ["T001","2026-05-01 09:10","C001","CH01","Доступ к аккаунту","Высокий","Resolved",2.5],
    ["T002","2026-05-01 10:25","C002","CH02","Оплата","Средний","Resolved",5.0],
    ["T003","2026-05-01 11:40","C003","CH03","Техническая ошибка","Высокий","In Progress",None],
    ["T004","2026-05-02 08:15"," C004 ","CH01","Доставка","Низкий","Resolved",8.0],
    ["T005","2026-05-02 13:30","C005","CH04","Возврат","Средний","Open",None],
    ["T006","2026-05-03 09:05","C006"," CH02 ","Оплата","Высокий","Resolved",3.5],
    ["T007","2026-05-03 12:20","C007","CH03","Консультация","Низкий","Resolved",1.0],
    ["T008","2026-05-04 14:00","C008","CH05","Техническая ошибка","Высокий","Resolved",12.0],
    ["T009","2026-05-04 15:45","C009","CH01","Доступ к аккаунту","Средний","Resolved",4.0],
    ["T010","2026-05-05 09:35","C010","CH02","Доставка","Низкий","Open",None],
    ["T011","2026-05-05 11:10","C001","CH03","Оплата","Средний","Resolved",6.5],
    ["T012","2026-05-06 10:00","C002","CH04","Возврат","Высокий","Resolved",15.0],
    ["T013","2026-05-06 16:25","C003","CH05","Консультация","Низкий","Resolved",0.8],
    ["T014","2026-05-07 08:50","C004","CH01","Техническая ошибка","Высокий","In Progress",None],
    ["T015","2026-05-07 13:15","C005","CH02","Доступ к аккаунту","Средний","Resolved",2.0],
    ["T016","2026-05-08 09:45","C006","CH03","Доставка","Низкий","Resolved",7.0],
    ["T017","2026-05-08 12:35","C007","CH04","Оплата","Средний","Resolved",4.5],
    ["T018","2026-05-09 10:20","C008","CH05","Возврат","Высокий","Open",None],
    ["T019","2026-05-09 14:30","C009","CH01","Консультация","Низкий","Resolved",1.5],
    ["T020","2026-05-10 09:00","C010","CH02","Техническая ошибка","Высокий","Resolved",10.0],
    ["T021","2026-05-10 11:55","C001","CH03","Доставка","Средний","Resolved",6.0],
    ["T022","2026-05-11 08:40","C002","CH04","Доступ к аккаунту","Высокий","Resolved",3.0],
    ["T023","2026-05-11 13:20","C003","CH05","Оплата","Средний","In Progress",None],
    ["T024","2026-05-12 10:15","C004","CH01","Возврат","Низкий","Resolved",9.0],
    ["T025","2026-05-12 15:05","C005","CH02","Консультация","Низкий","Resolved",1.2],
    ["T026","2026-05-13 09:25","C006","CH03","Техническая ошибка","Высокий","Resolved",11.0],
    ["T027","2026-05-13 12:10","C007","CH04","Доставка","Средний","Resolved",5.5],
    ["T028","2026-05-14 08:30","C999","CH05","Доступ к аккаунту","Высокий","Resolved",2.8],
    ["T029","2026-05-14 14:45","C008","CH99","Оплата","Средний","Resolved",4.2],
    ["T030","2026-05-15 10:05","C999","CH01","Возврат","Высокий","Open",None],
]

clients_rows = [
    ["C001","B2C","Москва","2025-01-15"],
    ["C002","B2B","Санкт-Петербург","2025-02-20"],
    ["C003","B2C","Казань","2025-03-05"],
    ["C004","B2B","Екатеринбург","2025-03-18"],
    ["C005","B2C","Москва","2025-04-01"],
    ["C006","B2G","Новосибирск","2025-04-14"],
    ["C007","B2C","Самара","2025-05-02"],
    ["C008","B2B","Москва","2025-05-20"],
    ["C009","B2C","Ростов-на-Дону","2025-06-10"],
    ["C010","B2G","Нижний Новгород","2025-06-28"],
]

channels_rows = [
    ["CH01","Телефон",False],
    ["CH02","Электронная почта",True],
    ["CH03","Чат",True],
    ["CH04","Мобильное приложение",True],
    ["CH05","Офис",False],
]

if not TICKETS_PATH.exists():
    pd.DataFrame(
        tickets_rows,
        columns=["ticket_id","created_at","client_id","channel_code","category","priority","status","resolution_hours"]
    ).to_csv(TICKETS_PATH, index=False, encoding="utf-8-sig")
    print("Создан:", TICKETS_PATH)

if not CLIENTS_PATH.exists():
    pd.DataFrame(
        clients_rows,
        columns=["client_id","client_segment","city","registration_date"]
    ).to_excel(CLIENTS_PATH, index=False, sheet_name="clients")
    print("Создан:", CLIENTS_PATH)

if not CHANNELS_PATH.exists():
    pd.DataFrame(
        channels_rows,
        columns=["channel_code","channel_name","is_digital"]
    ).to_json(CHANNELS_PATH, orient="records", force_ascii=False, indent=2)
    print("Создан:", CHANNELS_PATH)

### Проверяем файлы

In [ ]:
required_files = [TICKETS_PATH, CLIENTS_PATH, CHANNELS_PATH]

print("Проверяем исходные файлы:")
for file_path in required_files:
    status = "OK" if file_path.exists() else "НЕ НАЙДЕН"
    print(f"{status}: {file_path}")

assert all(path.exists() for path in required_files), "Не все исходные файлы доступны."

## 3. Загружаем CSV

CSV — текстовый табличный формат. После загрузки обязательно проверяем:

1. первые строки;
2. размер таблицы;
3. названия столбцов;
4. типы данных;
5. пропуски и дубликаты.

In [ ]:
# dtype помогает сразу сохранить идентификаторы как текст.
tickets = pd.read_csv(
    TICKETS_PATH,
    dtype={"ticket_id": "string", "client_id": "string", "channel_code": "string"}
)

display(tickets.head())
print("Размер tickets:", tickets.shape)

### Первичная диагностика таблицы обращений

In [ ]:
print("Столбцы:")
print(tickets.columns.tolist())

print("\nТипы данных:")
display(tickets.dtypes.to_frame("dtype"))

print("\nПропуски:")
display(tickets.isna().sum().to_frame("missing_values"))

print("\nДубликаты ticket_id:", tickets["ticket_id"].duplicated().sum())

### Преобразуем дату

После чтения CSV дата часто имеет тип `object`, то есть обычный текст.

In [ ]:
# errors="coerce" превращает нераспознанные значения в NaT.
tickets["created_at"] = pd.to_datetime(tickets["created_at"], errors="coerce")

print("Тип created_at:", tickets["created_at"].dtype)
print("Нераспознанных дат:", tickets["created_at"].isna().sum())

**Контрольный вопрос.** Почему нельзя ограничиться только `head()`?

Запишите ответ в Markdown-ячейке: первые строки не показывают все типы ошибок, пропуски и дубликаты.

## 4. Загружаем Excel

В Excel может быть несколько листов, поэтому имя листа указываем явно.

In [ ]:
clients = pd.read_excel(
    CLIENTS_PATH,
    sheet_name="clients",
    dtype={"client_id": "string"}
)
clients["registration_date"] = pd.to_datetime(clients["registration_date"], errors="coerce")

display(clients.head())
print("Размер clients:", clients.shape)
print("Дубликаты client_id:", clients["client_id"].duplicated().sum())

## 5. Загружаем JSON

В этом занятии JSON имеет простую структуру: список одноуровневых объектов.

In [ ]:
channels = pd.read_json(CHANNELS_PATH, dtype={"channel_code": "string"})

display(channels)
print("Размер channels:", channels.shape)
print("Дубликаты channel_code:", channels["channel_code"].duplicated().sum())

## 6. Подготавливаем ключи

Ключ — поле, по которому строки одной таблицы связываются со строками другой таблицы.

Перед объединением проверяем:

- одинаковый смысл полей;
- совместимые типы;
- отсутствие лишних пробелов;
- уникальность ключа в справочнике.

In [ ]:
# Удаляем пробелы по краям. Это важно: " C004 " и "C004" — разные строки.
for column in ["client_id", "channel_code"]:
    tickets[column] = tickets[column].astype("string").str.strip()

clients["client_id"] = clients["client_id"].astype("string").str.strip()
channels["channel_code"] = channels["channel_code"].astype("string").str.strip()

print("Пример client_id после очистки:", tickets.loc[tickets["ticket_id"] == "T004", "client_id"].iloc[0])
print("Пример channel_code после очистки:", tickets.loc[tickets["ticket_id"] == "T006", "channel_code"].iloc[0])

### Ищем значения, которых нет в справочниках

In [ ]:
unknown_clients_before_merge = sorted(
    set(tickets["client_id"].dropna()) - set(clients["client_id"].dropna())
)
unknown_channels_before_merge = sorted(
    set(tickets["channel_code"].dropna()) - set(channels["channel_code"].dropna())
)

print("Клиенты, которых нет в справочнике:", unknown_clients_before_merge)
print("Каналы, которых нет в справочнике:", unknown_channels_before_merge)

## 7. Объединяем таблицы

Используем `left merge`, потому что главная сущность — обращение. Мы хотим сохранить все обращения, даже если справочник не содержит нужного клиента или канала.

In [ ]:
# left означает: сохраняем все обращения из основной таблицы tickets.
tickets_with_clients = tickets.merge(
    clients,
    on="client_id",
    how="left",
    validate="many_to_one"
)

print("Строк до объединения:", len(tickets))
print("Строк после добавления клиентов:", len(tickets_with_clients))
display(tickets_with_clients.head())

In [ ]:
tickets_datamart = tickets_with_clients.merge(
    channels,
    on="channel_code",
    how="left",
    validate="many_to_one"
)

print("Строк после добавления каналов:", len(tickets_datamart))
print("Столбцов в итоговой таблице:", tickets_datamart.shape[1])
display(tickets_datamart.head())

## 8. Проверяем качество объединения

Успешное выполнение кода ещё не доказывает корректность результата. После `merge` проверяем:

- не изменилось ли количество строк;
- сохранилась ли уникальность `ticket_id`;
- появились ли пропуски в полях из справочников;
- какие именно ключи не совпали.

In [ ]:
quality_report = pd.DataFrame({
    "check": [
        "Строк в исходной таблице",
        "Строк в итоговой таблице",
        "Уникальных ticket_id",
        "Обращений без найденного клиента",
        "Обращений без найденного канала",
        "Нераспознанных дат"
    ],
    "value": [
        len(tickets),
        len(tickets_datamart),
        tickets_datamart["ticket_id"].nunique(),
        tickets_datamart["client_segment"].isna().sum(),
        tickets_datamart["channel_name"].isna().sum(),
        tickets_datamart["created_at"].isna().sum()
    ]
})

display(quality_report)

assert len(tickets_datamart) == len(tickets), "После merge изменилось количество строк."
assert tickets_datamart["ticket_id"].nunique() == len(tickets), "ticket_id перестал быть уникальным."

In [ ]:
print("Обращения с неизвестным клиентом:")
display(
    tickets_datamart.loc[
        tickets_datamart["client_segment"].isna(),
        ["ticket_id", "client_id", "category", "status"]
    ]
)

print("Обращения с неизвестным каналом:")
display(
    tickets_datamart.loc[
        tickets_datamart["channel_name"].isna(),
        ["ticket_id", "channel_code", "category", "status"]
    ]
)

## 9. Небольшая аналитика

Проверим, что единая таблица уже подходит для простых расчётов.

In [ ]:
print("Количество обращений по каналам:")
display(
    tickets_datamart["channel_name"]
    .fillna("Неизвестный канал")
    .value_counts()
    .rename_axis("channel_name")
    .reset_index(name="tickets_count")
)

print("Среднее время решения по сегментам:")
display(
    tickets_datamart
    .groupby("client_segment", dropna=False, as_index=False)
    .agg(
        tickets_count=("ticket_id", "nunique"),
        avg_resolution_hours=("resolution_hours", "mean")
    )
    .sort_values("avg_resolution_hours", ascending=False)
)

## 10. Сохраняем результат

Используем `index=False`, чтобы pandas не добавил технический столбец с номерами строк.

In [ ]:
tickets_datamart.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
print("Итоговый файл сохранён:", OUTPUT_PATH)
print("Размер файла, байт:", OUTPUT_PATH.stat().st_size)

### Повторно читаем сохранённый файл

Это простая проверка того, что файл действительно создан и открывается.

In [ ]:
result_check = pd.read_csv(OUTPUT_PATH)
print("Размер повторно загруженного файла:", result_check.shape)
display(result_check.head())

assert result_check.shape == tickets_datamart.shape, "Размер сохранённого файла отличается от DataFrame."

## 11. Мини-задание

Подготовьте отдельную таблицу по решённым обращениям. Подсказка: понадобятся `loc`, `groupby`, `agg`, `sort_values` и `to_csv`.

In [ ]:
# Мини-задание для самостоятельной работы.
# Выполните его в новой ячейке ниже или замените None своим результатом.
#
# 1. Оставьте только обращения со статусом Resolved.
# 2. Сгруппируйте их по client_segment.
# 3. Рассчитайте количество обращений и среднее resolution_hours.
# 4. Отсортируйте по среднему времени решения по убыванию.
# 5. Сохраните результат в outputs/resolved_tickets_by_segment.csv.

resolved_by_segment = None

print("Задание подготовлено. Создайте решение в следующей ячейке.")

### Место для решения

Добавьте новую кодовую ячейку под этой ячейкой и выполните мини-задание.

После расчёта сформулируйте один вывод:

> Какой сегмент имеет максимальное среднее время решения среди решённых обращений?

## 12. Типовые ошибки

| Ошибка | Причина | Что сделать |
|---|---|---|
| `FileNotFoundError` | Не выполнена подготовка папок или открыт другой каталог | Повторно выполнить блоки 2–3 и посмотреть `Path.cwd()` |
| CSV открылся одной колонкой | Указан неверный разделитель | Проверить параметр `sep` |
| Не читается XLSX | Нет `openpyxl` | Установить библиотеку или выполнить `pip install openpyxl` |
| Дата осталась текстом | Формат не преобразован | Использовать `pd.to_datetime()` |
| Ошибка типов при `merge` | Один ключ — число, другой — строка | Привести оба ключа к `string` |
| После `merge` появились пропуски | Ключ отсутствует в справочнике | Найти строки через `isna()` |
| После `merge` стало больше строк | Ключ справочника неуникален | Проверить `duplicated()` |

## 13. Контрольные вопросы

1. Чем CSV отличается от Excel с точки зрения загрузки?
2. Что показывает `shape`?
3. Почему даты могут загрузиться как текст?
4. Что называется ключом объединения?
5. Почему используется `how="left"`?
6. Что проверяет параметр `validate="many_to_one"`?
7. Почему после объединения важно сравнить число строк?
8. Зачем повторно читать сохранённый файл?

## 14. Чек-лист завершения

- [ ] Все ячейки основного маршрута выполнены сверху вниз.
- [ ] Три файла найдены или созданы.
- [ ] Даты преобразованы в `datetime`.
- [ ] Ключи очищены от пробелов.
- [ ] После объединения осталось 30 обращений.
- [ ] Найдены обращения с неизвестным клиентом и каналом.
- [ ] Создан `outputs/support_tickets_datamart.csv`.
- [ ] Итоговый файл повторно открывается.
- [ ] Выполнено мини-задание.
- [ ] Сформулирован аналитический вывод.

## Итог

Вы прошли полный базовый цикл интеграции данных:

**файлы → загрузка → проверка → подготовка ключей → объединение → контроль качества → сохранение результата**.

На следующем занятии эта логика будет расширена типами соединений, контролем гранулярности и диагностикой размножения строк.